In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "db" / "retail.db"
REPORTS_DIR = PROJECT_ROOT / "reports"

conn = sqlite3.connect(DB_PATH)

daily_raw = pd.read_sql("""
SELECT o.store_id,
       DATE(o.order_ts) AS date,
       SUM(oi.line_revenue) AS sales,
       COUNT(DISTINCT o.order_id) AS orders
FROM clean_order_items AS oi
JOIN clean_orders AS o ON o.order_id = oi.order_id
GROUP BY o.store_id, DATE(o.order_ts)
""", conn, parse_dates=["date"])

print(daily_raw.shape)
daily_raw.head()

(8295, 4)


,store_id,date,sales,orders
0,S001,2024-01-17,5745.0,1
1,S001,2024-01-21,120.0,1
2,S001,2024-01-22,4188.5,1
3,S001,2024-01-27,22095.5,3
4,S001,2024-01-28,3492.0,1


In [2]:
stores = pd.read_sql("SELECT store_id, store_type, region FROM clean_stores", conn)
all_dates = pd.date_range("2024-01-01", "2025-12-31", freq="D")

grid = pd.MultiIndex.from_product([stores["store_id"], all_dates],
                                  names=["store_id", "date"]).to_frame(index=False)

daily = grid.merge(daily_raw, on=["store_id", "date"], how="left")
print("Store-days with no sales:", daily["sales"].isna().sum())

daily[["sales", "orders"]] = daily[["sales", "orders"]].fillna(0)
daily = daily.merge(stores, on="store_id", how="left")
daily = daily.sort_values(["store_id", "date"]).reset_index(drop=True)

print(daily.shape)
daily.head()

Store-days with no sales: 477
(8772, 6)


,store_id,date,sales,orders,store_type,region
0,S001,2024-01-01,0.0,0.0,Express,South
1,S001,2024-01-02,0.0,0.0,Express,South
2,S001,2024-01-03,0.0,0.0,Express,South
3,S001,2024-01-04,0.0,0.0,Express,South
4,S001,2024-01-05,0.0,0.0,Express,South


In [3]:
FESTIVE_MONTHS = [10, 11]
STORE_TYPE_CODES = {"Express": 0, "Standard": 1, "Flagship": 2}

daily["day_of_week"] = daily["date"].dt.day_of_week
daily["is_weekend"] = (daily["day_of_week"] >= 5).astype(int)
daily["month"] = daily["date"].dt.month
daily["day_of_month"] = daily["date"].dt.day
daily["is_festive_season"] = daily["month"].isin(FESTIVE_MONTHS).astype(int)
daily["store_type_code"] = daily["store_type"].map(STORE_TYPE_CODES)

daily[["store_id", "date", "sales", "day_of_week", "is_weekend", "month",
       "is_festive_season", "store_type_code"]].head(8)

,store_id,date,sales,day_of_week,is_weekend,month,is_festive_season,store_type_code
0,S001,2024-01-01,0.0,0,0,1,0,0
1,S001,2024-01-02,0.0,1,0,1,0,0
2,S001,2024-01-03,0.0,2,0,1,0,0
3,S001,2024-01-04,0.0,3,0,1,0,0
4,S001,2024-01-05,0.0,4,0,1,0,0
5,S001,2024-01-06,0.0,5,1,1,0,0
6,S001,2024-01-07,0.0,6,1,1,0,0
7,S001,2024-01-08,0.0,0,0,1,0,0


In [4]:
HORIZON = 7

store_sales = daily.groupby("store_id")["sales"]

daily["sales_lag_7"] = store_sales.shift(7)
daily["sales_lag_14"] = store_sales.shift(14)
daily["sales_lag_28"] = store_sales.shift(28)

daily["sales_mean_7d"] = store_sales.transform(lambda s: s.shift(HORIZON).rolling(7).mean())
daily["sales_mean_28d"] = store_sales.transform(lambda s: s.shift(HORIZON).rolling(28).mean())

daily[daily["store_id"] == "S009"][["date", "sales", "sales_lag_7", "sales_mean_7d"]].iloc[30:38]

,date,sales,sales_lag_7,sales_mean_7d
5878,2024-01-31,22928.0,6375.0,4159.214286
5879,2024-02-01,8059.5,13065.5,6025.714286
5880,2024-02-02,13902.0,0.0,6025.714286
5881,2024-02-03,66263.0,46662.0,11507.642857
5882,2024-02-04,24513.0,64574.5,20732.571429
5883,2024-02-05,12960.0,0.0,19416.571429
5884,2024-02-06,0.0,0.0,18668.142857
5885,2024-02-07,10654.0,22928.0,21032.857143


In [5]:
store_id = "S009"
target_date = pd.Timestamp("2025-06-15")

history = daily[daily["store_id"] == store_id].set_index("date")["sales"]
row = daily[(daily["store_id"] == store_id) & (daily["date"] == target_date)].iloc[0]

last_known_day = target_date - pd.Timedelta(days=HORIZON)
print("Predicting:", target_date.date(), "| last day we're allowed to know:", last_known_day.date())

by_hand = {
    "sales_lag_7": history[target_date - pd.Timedelta(days=7)],
    "sales_lag_14": history[target_date - pd.Timedelta(days=14)],
    "sales_lag_28": history[target_date - pd.Timedelta(days=28)],
    "sales_mean_7d": history[last_known_day - pd.Timedelta(days=6): last_known_day].mean(),
    "sales_mean_28d": history[last_known_day - pd.Timedelta(days=27): last_known_day].mean(),
}

for feature, expected in by_hand.items():
    match = abs(row[feature] - expected) < 0.01
    print(f"{feature:<16} code={row[feature]:>12.2f}  by hand={expected:>12.2f}  match={match}")
    assert match, f"{feature} does not match: possible leakage"

print("\nNo leakage: every feature uses only data from", last_known_day.date(), "or earlier.")

Predicting: 2025-06-15 | last day we're allowed to know: 2025-06-08
sales_lag_7      code=    98852.50  by hand=    98852.50  match=True
sales_lag_14     code=   223348.50  by hand=   223348.50  match=True
sales_lag_28     code=   240609.50  by hand=   240609.50  match=True
sales_mean_7d    code=   146612.43  by hand=   146612.43  match=True
sales_mean_28d   code=   156921.88  by hand=   156921.88  match=True

No leakage: every feature uses only data from 2025-06-08 or earlier.


In [7]:
FEATURE_COLUMNS = ["day_of_week", "is_weekend", "month", "day_of_month", "is_festive_season",
                   "store_type_code", "sales_lag_7", "sales_lag_14", "sales_lag_28",
                   "sales_mean_7d", "sales_mean_28d"]

features = daily.dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)
print("Rows with full history:", len(features), "| first date:", features["date"].min().date())

features.to_sql("features_daily_sales", conn, if_exists="replace", index=False)

catalog = pd.DataFrame([
    ("day_of_week", "Calendar", "0 = Monday ... 6 = Sunday", "Date is known in advance"),
    ("is_weekend", "Calendar", "1 on Saturday/Sunday", "Date is known in advance"),
    ("month", "Calendar", "1-12", "Date is known in advance"),
    ("day_of_month", "Calendar", "1-31", "Date is known in advance"),
    ("is_festive_season", "Calendar", "1 in October-November", "Date is known in advance"),
    ("store_type_code", "Store", "Express 0, Standard 1, Flagship 2", "Store type is fixed"),
    ("sales_lag_7", "Recent sales", "Sales on the same weekday 1 week earlier", "Exactly 7 days back"),
    ("sales_lag_14", "Recent sales", "Sales on the same weekday 2 weeks earlier", "14 days back"),
    ("sales_lag_28", "Recent sales", "Sales on the same weekday 4 weeks earlier", "28 days back"),
    ("sales_mean_7d", "Recent sales", "Average daily sales over the 7 days ending 7 days before",
     "Window ends 7 days back (shift before rolling)"),
    ("sales_mean_28d", "Recent sales", "Average daily sales over the 28 days ending 7 days before",
     "Window ends 7 days back (shift before rolling)"),
], columns=["feature", "group", "description", "why_available_at_prediction_time"])

catalog.to_sql("feature_catalog", conn, if_exists="replace", index=False)
catalog.to_csv(REPORTS_DIR / "feature_catalog.csv", index=False)

catalog

Rows with full history: 8364 | first date: 2024-02-04


,feature,group,description,why_available_at_prediction_time
0,day_of_week,Calendar,0 = Monday ... 6 = Sunday,Date is known in advance
1,is_weekend,Calendar,1 on Saturday/Sunday,Date is known in advance
2,month,Calendar,1-12,Date is known in advance
3,day_of_month,Calendar,1-31,Date is known in advance
4,is_festive_season,Calendar,1 in October-November,Date is known in advance
5,store_type_code,Store,"Express 0, Standard 1, Flagship 2",Store type is fixed
6,sales_lag_7,Recent sales,Sales on the same weekday 1 week earlier,Exactly 7 days back
7,sales_lag_14,Recent sales,Sales on the same weekday 2 weeks earlier,14 days back
8,sales_lag_28,Recent sales,Sales on the same weekday 4 weeks earlier,28 days back
9,sales_mean_7d,Recent sales,Average daily sales over the 7 days ending 7 d...,Window ends 7 days back (shift before rolling)
